In [ ]:
import torch
from tqdm import tqdm
from sfm.data.librimix import LibriMix as DS
from sfm.data.base_datamodule import BaseDatamodule as DM
from sfm.src.base_experiment import BaseExp
from IPython.display import display, Audio
from itertools import islice
import yaml
import matplotlib.pyplot as plt

CONFIG_PATH = '../../config/'

In [ ]:
class IsotropicNoiseExp(BaseExp):
    def __init__(
            self,
            **parent_kwargs,
        ):
        super().__init__(
           **parent_kwargs
        )
        
    def get_spatial_coherence(
        self,
        noise_stft: torch.Tensor,  # (BATCH, CHANNEL, FREQ, FRAMES)
        mic_pos: torch.Tensor,  # (BATCH, CHANNEL, 3)
    ) -> torch.Tensor:  # (BATCH, CHANNEL, CHANNEL, FREQ)
        n_batch, n_channel, n_freq, n_frames = noise_stft.shape
        
        # noise coherence
        noise_psd = torch.einsum(
            '...ijk, ...ljk -> ...iljk', noise_stft, torch.conj(noise_stft)
        ).mean(-1)  # (BATCH, CHANNEL, CHANNEL, FREQ)
        noise_power = noise_psd.diagonal(dim1=-3, dim2=-2).real.swapdims(-1, -2)  # (BATCH, CHANNEL, FREQ)
        noise_coherence_den = torch.einsum(
            '...ij, ...kj -> ...ikj', noise_power, noise_power
        ).sqrt()  # (BATCH, CHANNEL, CHANNEL, FREQ)
        noise_coherence = noise_psd / noise_coherence_den.clamp(min=1e-9)
        
        # ideal coherence
        distance = torch.linalg.norm(
            mic_pos[:, None] - mic_pos[:, :, None], ord=2, dim=-1
        ) / 343 * self.sample_rate  # (BATCH, CHANNEL, CHANNEL)
        ideal_coherence = torch.sinc(
            2 * torch.arange(n_freq, device=self.device) / self.stft_length * distance[..., None]
        )  # (BATCH, CHANNEL, CHANNEL, FREQ)
        
        return noise_coherence, ideal_coherence

    def forward(self, batch):
        # compute stft and diffuse noise
        self.preprocessing(batch)
       
        # compute spatial coherence
        noisy_coherence, ideal_coherence = self.get_spatial_coherence(
            batch['noise_stft'], batch['mic_pos']
        )  # (BATCH, CHANNEL, FREQ, FRAMES)

        # get TD
        self.postprocessing(batch)
        
        return batch['mix_td'], batch['noisy_td'], noisy_coherence, ideal_coherence

In [ ]:
# setup experiment
data_config_path = CONFIG_PATH + 'base_config.yaml'
with open(data_config_path, 'r') as file:
    config = yaml.safe_load(file)
dm = DM(
    DS,
    batch_size=1,
    n_workers=0,
    **config
)
exp = IsotropicNoiseExp(
    model=None,
    **config, 
)

In [ ]:
n_batch = 5
mix_td, noisy_td, noisy_coherence, ideal_coherence = [], [], [], []
for example in islice(dm.val_dataloader(), n_batch):
    
    # forward noise computation
    m_td, n_td, n_coherence, i_coherence = exp(example)
    mix_td.append(m_td.squeeze(0).cpu())
    noisy_td.append(n_td.squeeze(0).cpu())
    noisy_coherence.append(n_coherence.squeeze(0).cpu())
    ideal_coherence.append(i_coherence.squeeze(0).cpu())

In [ ]:
batch_idx = 4
fig, axs = plt.subplots(ncols=exp.n_channel-1)
for c in range(exp.n_channel-1):
    axs[c].plot(noisy_coherence[batch_idx][0, c+1].real)
    axs[c].plot(ideal_coherence[batch_idx][0, c+1].real)